In [ ]:
import marimo as mo

# 🧪Lab: Predicting Seismic Building Response with Support Vector Regression

In this lab, you will practice **Support Vector Machine (SVM)** in the context of **regression**. Over the past two weeks, we have focused on SVM for **classification problems**. However, SVM can also be extended to handle **regression tasks**—this is known as **Support Vector Regression (SVR)**.

To explore this, you will work with the following paper:  https://www.nature.com/articles/s41598-024-81705-3

In this paper, the authors use **real seismic monitoring data** to develop an SVR-based model for predicting the **maximum inter-story drift ratio** of buildings during earthquakes. The authors compare a full-feature model with a reduced-feature version, and demonstrate that SVR is effective in this complex, real-world regression setting.

You will begin by reading the **introduction** of the paper to understand the motivation behind the problem. Later, you will apply Support Vector Regression to their dataset.

Reference:

- Tao, D., Fang, S., Liu, H. et al. Support vector regression model for the prediction of buildings' maximum seismic response based on real monitoring data. Sci Rep 14, 29874 (2024). https://doi.org/10.1038/s41598-024-81705-3

---

**Software**: Use `scikit-learn` and its implementation of Support Vector Regression in this lab. https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.

In [ ]:
GROUP = "group11-section1"

## 1. Paper reflection (10 points)

a. After reading the introduction of the paper, discuss within the group and address the following questions:

   - Why is predicting seismic building response important? (You may consider implications for human safety, economic losses, emergency planning, etc)

- Why might the dataset of this study be particularly well-suited for a machine learning approach? (You may think about the volume and diversity of data, the types of features provided, and the limitations of traditional physics-based methods).

Please, elaborate on your answers.

Predicting how a building responds matters because the response is what turns shaking into damage. The relevant quantities are the maximum inter-story drift ratio, peak floor acceleration and residual drift, and the introduction puts the cost of ground-shaking damage to US buildings at about 14.7 billion USD per year, on top of casualties and lost function. After a large earthquake a city can have hundreds of thousands of buildings, most with incomplete design records, and emergency crews and loss assessors need to know quickly which ones took the worst hit. In that situation a fast prediction that is roughly right beats a slow simulation that is exact.

The dataset suits machine learning for practical reasons. NDE1.0 has roughly 12,000 recordings from real instrumented buildings (steel frame, RC, SRC, masonry, wood) across thousands of earthquakes, collected from the CESMD and BRI monitoring networks, so there is plenty of training data covering many building types and shaking levels. Each record also carries 41 numeric parameters: building geometry and frequency, spectral quantities, durations, and intensities like PGA, PGV, PGD and CAV. Those are cheap to measure and give a model a lot of signal to work with. The usual alternative is weak here: a mechanics-based approach needs a finite-element model of each building plus the ground motion history recorded at its base, which most buildings lack and which is far too slow for near-real-time assessment of a whole city. Learning the mapping from measured parameters straight to measured response avoids that bottleneck, at the cost of some interpretability.

b. After reading the paper, explain, in your own words, how Support Vector Regression (SVR) works. In addition, explain Formulas (1) to (11) from the paper, specifically, what is the intuition behind each equation in light of what we covered in class regarding Support Vector Machine, so you can see the correspondence between SVM for classification and regression.

In classification SVM we looked for a separating hyperplane with the widest margin, and only the points on the margin mattered. SVR applies the same idea to regression: instead of a boundary between classes, it fits a function through the data and wants the widest tube of half-width $\epsilon$ around that function that still contains the training points. Points inside the tube have zero loss, so the fit is driven only by the points that escape it (the support vectors), and C sets the price of escaping. A kernel replaces inner products of transformed features, so the machinery that gave nonlinear boundaries in classification gives nonlinear regression here. The paper uses the Gaussian (RBF) kernel.:

1. Formula (1): $|W^T \cdot \Phi(x_i) + b - y_i| \le \epsilon$, for $i = 1, 2, ..., n$. Subtracting the true target values from the output of the linear regression function $f(x) = W^T \cdot \Phi(x_i) + b$, then taking the absolute value of that difference to determine the true error; the epsilon value is the permissible amount of error. This is the tube: a prediction within $\epsilon$ of the truth counts as a good fit, the regression version of the classification margin. $W$ is the weighted combination of individual parameters, $\Phi(x_i)$ a fixed feature-space transformation of the input and $b$ the intercept.

2. Formula (2): the regularized objective, minimize $\tfrac{1}{2} \sum_i^n E_i + \tfrac{\lambda}{2} \|W\|^2$. This matches the maximum-margin objective from class: among all functions that keep errors small, prefer the one with the smallest weights, i.e. the flattest function. $\lambda$ is a penalty parameter.

3. Formula (3): $E_i$ is defined piecewise: $E_i = 0$ if $|f(x_i) - y_i| \le \epsilon$, and $E_i = f(x_i) - y_i - \epsilon$ otherwise. The error only counts the part of the miss that sticks out of the tube; a point inside the tube costs nothing, like a correctly classified point beyond the margin in SVM.

4. Formula (4): minimize $\tfrac{1}{2}\|W\|^2 + C\sum_i^n (\xi_i + \xi_i^*)$, rewriting with upper and lower slack variables $\xi_i, \xi_i^*$ (one per side of the tube). This is the soft-margin formulation from class: $C$ is a specified constant, the penalty parameter, representing the penalty degree of the sample when it exceeds the permissible error. Small $C$ tolerates violations (smoother fit), large $C$ fights for every point that escapes the tube.

5. Formula (5): the constraints, $f(x_i) - y_i \le \epsilon + \xi_i$ and $y_i - f(x_i) \le \epsilon + \xi_i^*$, with $\xi_i, \xi_i^* \ge 0$. The slack variables represent the training errors on the upper and lower sides of the regression function respectively, so these define how far beyond $\epsilon$ each point misses on each side.

6. Formula (6): the Lagrangian $L(W, b, \xi, \xi^*, \alpha, \beta, \gamma, \gamma^*)$. The constrained problem becomes unconstrained by attaching Lagrange multipliers: $\alpha_i, \beta_i$ for the two tube constraints and $\gamma_i, \gamma_i^*$ for the slacks, which is the move we used for the classification dual.

7. Formula (7): setting the derivatives of the Lagrangian with respect to $W$, $b$, $\xi$ and $\xi^*$ to zero gives the first stationarity result, $W = \sum_i^n (\alpha_i - \beta_i) \Phi(x_i)$. The solution is a linear combination of training points, which is what later makes kernels possible.

8. Formula (8): the second stationarity result, $\sum_i^n (\alpha_i - \beta_i) = 0$.

9. Formula (9): the third stationarity result, $\alpha_i + \gamma_i = \beta_i + \gamma_i^* = C$. This ties the multipliers to the box constraint $0 \le \alpha_i, \beta_i \le C$ (the slack multipliers $\gamma$ are nonnegative) and shows $C$ bounds how much weight any single point can get, exactly like C in the classification dual.

10. Formula (10): the intercept, $b = y_j - W^T \cdot \Phi(x_j) + \epsilon = y_j - \sum_i^n (\alpha_i - \beta_i) k(x_i, x_j) + \epsilon$, where $k(x_i, x_j) = \Phi^T(x_i)\Phi(x_j)$ is the kernel function and $j$ is a support vector ($\alpha_i > 0$). Any support vector on the tube boundary pins down $b$; in practice these values get averaged, but the formula shows $b$ comes only from support vectors.

11. Formula (11): the prediction, $f(x_j) = \sum_i^n (\alpha_i - \beta_i) k(x_i, x_j) + b$. Because $W$ is a combination of training points (formula 7), we never need $\Phi$ explicitly; the kernel applies the feature-space transformation implicitly. With the RBF kernel this lets SVR fit a nonlinear surface while keeping the margin-plus-slacks logic of classification.

## 2. Exploratory Analysis (15 Points)

a. Load the dataset accompanying this lab and create a new dataset containing the same input features used as predictors in the paper, along with the target variable, *Drift*. **You will use only this new dataset throughout the remainder of the lab.** Display the first few rows of the resulting dataset.

*Hint*: The exact input features used in the predictive framework are clearly specified in the paper. You may also refer to Table 3 to help identify the corresponding features in the provided dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from sklearn.base import clone
from sklearn.linear_model import LassoCV, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

In [ ]:
# 41 paper features (6 structure + 9 spectrum + 16 duration + 10 other) + Drift
raw = pd.read_excel("data.xlsx", header=[1, 2])
raw.columns = [" ".join(c).strip() for c in raw.columns]
rename = {
    "B_Lat. [degree]": "B_Lat", "B_Long. [degree]": "B_Long", "Height [cm]": "H",
    "No. of story Unnamed: 7_level_2": "N", "F1 Hz": "F1", "F2 Hz": "F2",
    "SA1 [cm/s2]": "Sa1", "SV1 [cm/s]": "Sv1", "SD1 [cm]": "Sd1",
    "SA2 [cm/s2]": "Sa2", "SV2 [cm/s]": "Sv2", "SD2 [cm]": "Sd2",
    "Avg_Sa [cm/s2]": "Avg_Sa", "Avg_Sv [cm/s]": "Avg_Sv", "Avg_Sd [cm]": "Avg_Sd",
    "Effective [sec]": "Effective", "Bracketed 1 [0.05g] [sec]": "Bracketed_1_0.05g",
    "Uniform [0.05g] [sec]": "Uniform_0.05g", "Bracketed [0.1g] [sec]": "Bracketed_0.1g",
    "Uniform [0.1g] [sec]": "Uniform_0.1g", "Bracketed [0.15g] [sec]": "Bracketed_0.15g",
    "Uniform [0.15g] [sec]": "Uniform_0.15g", "Bracketed [0.2g] [sec]": "Bracketed_0.2g",
    "Uniform [0.2g] [sec]": "Uniform_0.2g",
    "Significant_a1 [(5-75)%] [sec]": "Sig_a1", "Significant_a2 [(5-95)%] [sec]": "Sig_a2",
    "Significant_v1 [(5-75)%] [sec]": "Sig_v1", "Significant_v2 [(5-95)%] [sec]": "Sig_v2",
    "Significant_d1 [(5-75)%] [sec]": "Sig_d1", "Significant_d2 [(5-95)%] [sec]": "Sig_d2",
    "ZX [sec]": "ZX",
    "E_Lat. [degree]": "E_Lat", "E_Long. [degree]": "E_Long", "Magnitude [Mw]": "M",
    "Epicentral distance (R) [km]": "R", "PGA [cm/s2]": "PGA", "PGV [cm/s]": "PGV",
    "PGD [cm]": "PGD", "AI [cm/s]": "Ia", "CAV [cm/s]": "CAV", "DP [cm*s]": "DP",
    "Drift [cm/cm]": "Drift",
}
df = raw[list(rename)].rename(columns=rename)
df = df.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
FEATURES = [c for c in df.columns if c != "Drift"]
print(f"{df.shape[0]} complete records, {len(FEATURES)} features")
df.head()

b. Explore the distribution of the target variable (Drift), and create scatterplots between Drift and input features.

In [ ]:
print(df["Drift"].describe().round(3))

fig, axes = plt.subplots(6, 7, figsize=(26, 20))
for ax_s, f in zip(axes.ravel(), FEATURES):
    ax_s.scatter(df[f], df["Drift"], s=4, alpha=0.25)
    r = np.corrcoef(df[f], df["Drift"])[0, 1]
    ax_s.set_title(f"{f}  (r = {r:.2f})", fontsize=9)
    ax_s.tick_params(labelsize=7)
axes.ravel()[-1].axis("off")
fig.suptitle("Drift vs input features", fontsize=14)
plt.tight_layout()
plt.show()

fig2, ax_hist = plt.subplots(figsize=(7, 4))
ax_hist.hist(df["Drift"], bins=100)
ax_hist.set_yscale("log")
ax_hist.set_xlabel("Drift")
ax_hist.set_ylabel("count (log scale)")
plt.show()

c. Highlight any particular pattern you observe from the visualizations. Which input features seem potentially more influential in predicting Drift?

Drift is heavily right-skewed: most records sit below the mean and the tail runs out to about 41 standard deviations, so we plotted the histogram on a log count scale. There are also a handful of extreme outliers that no single feature explains.

In the scatterplots the ground-motion intensity measures dominate. Drift rises in a tight, nearly linear cloud with the velocity- and displacement-type parameters: Sv1 (r = 0.72), Avg_Sv (0.70), PGV (0.66), Sd1 (0.65), CAV (0.63), Sa1 (0.62), PGD (0.62) and PGA (0.60). The duration parameters (the bracketed/uniform durations at the 0.15g and 0.2g thresholds, r around 0.59) correlate moderately but fan out much more. Building and event coordinates (B_Lat, B_Long, E_Lat, E_Long) are basically flat, which makes sense since coordinates only proxy for everything else. F1, F2, story count and height show only weak trends on their own. Our shortlist for the most influential features would be Sv1, Avg_Sv, PGV, Sd1, CAV, Sa1 and PGD.

## 3. Fit and evaluate (30 points)

a. Fit and evaluate a predictive framework using an SVR model. The process should include automatic tuning of both the kernel and the penalty parameter ((C)). Tune the hyperparameters using cross-validation on the training set, and evaluate the final predictive framework on a separate reporting set. Evaluate its performance using at least the Coefficient of Determination ($R^2$) and Mean Squared Error (MSE).

*N.B.* Remember that a predictive framework may involve more than just the predictive algorithm itself. Consider whether the characteristics of the predictors require any additional steps before fitting the model.

In [ ]:
# scale the predictors (SVM is not scale invariant), 9:1 split like the paper
X = df[FEATURES].to_numpy()
y = df["Drift"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

# CV search on a 3000-point subsample of the training set for tractability;
# the final model is refit on the full training set
sub = np.random.RandomState(42).choice(len(X_train), 3000, replace=False)
svr_grid = [
    {"svr__kernel": ["rbf"], "svr__C": [1, 10, 100], "svr__gamma": ["scale", 0.1, 0.01]},
    {"svr__kernel": ["linear"], "svr__C": [0.1, 1, 10, 100]},
    {"svr__kernel": ["poly"], "svr__C": [1, 10], "svr__degree": [2, 3]},
]
svr_search = GridSearchCV(
    make_pipeline(StandardScaler(), SVR(epsilon=0.1)), svr_grid, cv=3,
    scoring="r2", n_jobs=-1,
).fit(X_train[sub], y_train[sub])

svr_model = clone(svr_search.best_estimator_).fit(X_train, y_train)
svr_pred = svr_model.predict(X_test)
svr_r2 = r2_score(y_test, svr_pred)
svr_mse = mean_squared_error(y_test, svr_pred)

print("best params:", svr_search.best_params_)
print(f"CV R2 (subsample): {svr_search.best_score_:.3f}")
print(f"reporting set: R2 = {svr_r2:.3f}, MSE = {svr_mse:.3f}")

b. Repeat the previous process using an Ridge regression predictive framework. Be sure to include automatic tuning of the model's hyperparameters, specifically, `alpha`.

In [ ]:
ridge_search = GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": np.logspace(-3, 4, 29)},
    cv=5,
    scoring="r2",
    n_jobs=-1,
).fit(X_train, y_train)

ridge_model = ridge_search.best_estimator_
ridge_pred = ridge_model.predict(X_test)
ridge_r2 = r2_score(y_test, ridge_pred)
ridge_mse = mean_squared_error(y_test, ridge_pred)

print(f"best alpha: {ridge_search.best_params_['ridge__alpha']:.4g}")
print(f"reporting set: R2 = {ridge_r2:.3f}, MSE = {ridge_mse:.3f}")

c. Use predicted vs. true value scatterplots to compare the SVR and Ridge predictive frameworks. Embed performance metrics (e.g., R², MSE) in each plot.

Which predictive framework appears to perform better at this task?

In [ ]:
fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(13, 6))
for ax, pred, name, r2, mse in (
    (axes_cmp[0], svr_pred, "SVR", svr_r2, svr_mse),
    (axes_cmp[1], ridge_pred, "Ridge", ridge_r2, ridge_mse),
):
    ax.scatter(y_test, pred, s=5, alpha=0.3)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("true Drift")
    ax.set_ylabel("predicted Drift")
    ax.set_title(f"{name}  (R2 = {r2:.3f}, MSE = {mse:.3f})")
    ax.grid(True)
plt.tight_layout()
plt.show()

The SVR framework performs better, though not by a lot. On the reporting set SVR reaches R² = 0.466 with MSE = 0.266, while Ridge gets R² = 0.437 with MSE = 0.280. Both plots show regression-to-the-mean behavior: the frameworks underpredict the 16 most extreme records, with mean error about -2.9 for SVR and -2.0 for Ridge. Keep in mind the CV tuner picked a linear kernel with small C = 0.1 for SVR, so this comparison is mostly between two linear fits, and SVR still comes out ahead on both metrics.

## 4. Reduced SVR Model (30 points)

a. Add a feature selection step to the SVR predictive framework that selects the 10 features most highly correlated with Drift. Retrain  this updated framework and evaluate its performance.

In [ ]:
# correlations computed on the training set only, to avoid leaking the reporting set
train_df = pd.DataFrame(X_train, columns=FEATURES)
train_df["Drift"] = y_train
corr = train_df.corr()["Drift"].drop("Drift")
top10 = list(corr.abs().sort_values(ascending=False).head(10).index)
print("top 10 by |correlation| with Drift:", top10)

idx = [FEATURES.index(f) for f in top10]
sub_red = np.random.RandomState(42).choice(len(X_train), 3000, replace=False)
svr_red_grid = [
    {"svr__kernel": ["rbf"], "svr__C": [1, 10, 100], "svr__gamma": ["scale", 0.1, 0.01]},
    {"svr__kernel": ["linear"], "svr__C": [0.1, 1, 10, 100]},
    {"svr__kernel": ["poly"], "svr__C": [1, 10], "svr__degree": [2, 3]},
]
svr_red_search = GridSearchCV(
    make_pipeline(StandardScaler(), SVR(epsilon=0.1)), svr_red_grid, cv=3,
    scoring="r2", n_jobs=-1,
).fit(X_train[sub_red][:, idx], y_train[sub_red])

svr_red_model = clone(svr_red_search.best_estimator_).fit(X_train[:, idx], y_train)
svr_red_pred = svr_red_model.predict(X_test[:, idx])
svr_red_r2 = r2_score(y_test, svr_red_pred)
svr_red_mse = mean_squared_error(y_test, svr_red_pred)

print("best params:", svr_red_search.best_params_)
print(f"reporting set: R2 = {svr_red_r2:.3f}, MSE = {svr_red_mse:.3f}")

Performance drops moderately: the full 41-feature SVR gets R² = 0.466 (MSE = 0.266) on the reporting set, the 10-feature version gets R² = 0.410 (MSE = 0.294). That is a loss of about 0.06 in R² (roughly 12% relative) for 31 fewer features. The paper's reduced model lost much less, which makes sense: their 10 features were picked for usefulness and availability via SHAP, while pure correlation keeps redundant velocity-type measures (Sv1, Avg_Sv and Sv2 are near-duplicates of each other) and our reduced search settled on an RBF kernel with gamma = 0.01 that fits worse than the full model. One thing worth flagging: our correlation-based top 10 keeps co-seismic quantities (Avg_Sv, Sv2) that the paper deliberately dropped, since those cannot be computed until the building has already responded.

The practical advantages hold anyway. Data collection gets cheaper and faster: the features that survive (peak ground quantities, magnitude, epicentral distance, building height, story count, fundamental frequency) are known from building inventories or quick post-event measurements, while the co-seismic ones require response records from inside the building. Training and prediction also get faster with 10 inputs instead of 41, and a shorter feature list is easier to explain to the engineers who have to act on the prediction.

b. Compare the performance of this reduced-feature SVR framework with an optimal Lasso regression framework that uses all available features.

- Do both frameworks select the same features?
- How do their performances compare?

In [ ]:
lasso_pipe = make_pipeline(
    StandardScaler(),
    LassoCV(cv=5, alphas=100, max_iter=100000, random_state=42),
).fit(X_train, y_train)
lasso = lasso_pipe.named_steps["lassocv"]
lasso_pred = lasso_pipe.predict(X_test)
lasso_r2 = r2_score(y_test, lasso_pred)
lasso_mse = mean_squared_error(y_test, lasso_pred)

nz = [f for f, c in zip(FEATURES, lasso.coef_) if c != 0]
overlap = sorted(set(nz) & set(top10))
print(f"best alpha: {lasso.alpha_:.4g}, nonzero features: {len(nz)} of {len(FEATURES)}")
print("lasso nonzero:", nz)
print("overlap with the 10 correlated features:", overlap)
print(f"reporting set: R2 = {lasso_r2:.3f}, MSE = {lasso_mse:.3f}")

The two frameworks overlap only partially. Lasso keeps 6 features with nonzero coefficients (Sv1, Sd1, PGD, Ia, DP, B_Long), and just 3 of them (PGD, Sd1, Sv1) are in the top-10 correlation set. The reason is that Lasso works on standardized features and keeps whichever variables carry unique signal after accounting for the others, while plain correlation rewards any variable that moves with Drift even when it is redundant with a stronger one. That is why the correlation list is full of near-duplicate velocity measures while Lasso picks roughly one representative per information channel; it even keeps B_Long, a coordinate, presumably as a proxy for region and network differences.

On performance, the full-feature Lasso reaches R² = 0.441 (MSE = 0.278), which lands between the two SVR frameworks: below the full-feature SVR (0.466) but above the reduced 10-feature SVR (0.410). So the reduced SVR does not beat Lasso here, but it needs far fewer inputs, and the full-feature SVR beats both. All three fits are mostly linear: the tuner chose a linear kernel for SVR, and Lasso is linear by construction, so the differences come from regularization and feature set rather than from nonlinearity.

## 5: Robustness to Point Removal (10 points)

a. Compare the sensitivity of SVR and Ridge Regression predictive frameworks to small changes in the training set. To achieve this, randomly remove 10 training points, retrain both predictive frameworks, and assess how much the predictions change.

- Which predictive frameworks appears more stable?
- Why?

In [ ]:
changes_svr, changes_ridge = [], []
for rep in range(20):
    drop = np.random.RandomState(42 + rep).choice(len(X_train), 10, replace=False)
    keep = np.setdiff1d(np.arange(len(X_train)), drop)
    p_s = clone(svr_model).fit(X_train[keep], y_train[keep]).predict(X_test)
    p_r = clone(ridge_model).fit(X_train[keep], y_train[keep]).predict(X_test)
    changes_svr.append(np.abs(p_s - svr_pred).mean())
    changes_ridge.append(np.abs(p_r - ridge_pred).mean())
    print(f"rep {rep}: SVR mean |pred change| = {changes_svr[-1]:.5f}, Ridge = {changes_ridge[-1]:.5f}")

print(f"\naverage over 20 removals: SVR = {np.mean(changes_svr):.5f}, Ridge = {np.mean(changes_ridge):.5f}")

Ridge came out slightly more stable in our run: averaged over 20 random 10-point removals, its reporting-set predictions move by about 0.0003, versus about 0.0007 for SVR. Both numbers are tiny (the target has unit variance), so neither framework is fragile, but Ridge moves less.

The explanation is in the tuned hyperparameters. The CV search gave Ridge a very large alpha (about 3162), so its coefficients are dominated by the shrinkage penalty rather than by any particular data point, and dropping 10 of the ~10,700 training points barely moves a penalty-dominated solution. The tuned SVR is a linear-kernel model whose solution is pinned by the support vectors sitting on the epsilon-tube boundary; when a removal run happens to drop a boundary point, the tube shifts and the predictions move a bit more (one of our repetitions hit an outlier and both frameworks jumped). The textbook insensitivity argument for SVR only covers points strictly inside the tube; the points on the tube surface carry the full penalty C, so they do influence the fit.

## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

Our main inefficiency was compute time on the SVR tuning. We first tried to run the full grid search (kernel, C, gamma, degree) with cross-validation on the whole training set, and the linear-kernel fits alone took minutes each, which blocked everyone who was waiting on section 3 and forced us to re-run the search more than once after small changes. It cost us a good part of a work session that we had planned to spend on sections 4 and 5. The fix we landed on, doing the CV search on a 3000-point subsample and refitting the winner on the full training set, worked and we should have started there. For the next lab we will prototype every search on a small subsample first and only scale up once we know the grid is right, and we will split the long-running cells across group members so one blocked run does not hold up the whole notebook.